In [1]:
import numpy as np
import os

# Load data from your dataset.txt file
file_path = '/content/dataset.txt'

# if not os.path.exists(file_path):
#     print(f"Error: {file_path} not found. Please add some text to dataset.txt")
# else:
with open(file_path, 'r', encoding='utf-8') as f:
    text_data = f.read()

print(f"Loaded dataset with {len(text_data)} characters.")

Loaded dataset with 1115393 characters.


In [2]:
def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * np.power(x, 3))))

def gelu_backward(x, dout):
    cdf = 0.5 * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * np.power(x, 3))))
    pdf = np.exp(-0.5 * np.power(x, 2)) / np.sqrt(2.0 * np.pi)
    return dout * (cdf + x * pdf)

class CharTokenizer:
    def __init__(self, data):
        chars = sorted(list(set(data)))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}

    def encode(self, s):
        return [self.stoi[c] for c in s]

    def decode(self, l):
        return ''.join([self.itos[i] for i in l])

In [3]:
class Dataset:
    def __init__(self, data, tokenizer, block_size):
        self.data = np.array(tokenizer.encode(data))
        self.block_size = block_size

    def get_batch(self, batch_size):
        ix = np.random.randint(0, len(self.data) - self.block_size, batch_size)
        x = np.stack([self.data[i : i + self.block_size] for i in ix])
        y = np.stack([self.data[i + 1 : i + self.block_size + 1] for i in ix])
        return x, y

In [4]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * np.sqrt(2.0 / (in_features + out_features))
        self.b = np.zeros((out_features,))
        self.x = None

    def forward(self, x):
        self.x = x
        return np.dot(x, self.W) + self.b

    def backward(self, dout):
        x_reshaped = self.x.reshape(-1, self.x.shape[-1])
        dout_reshaped = dout.reshape(-1, dout.shape[-1])
        self.dW = np.dot(x_reshaped.T, dout_reshaped)
        self.db = np.sum(dout_reshaped, axis=0)
        return np.dot(dout, self.W.T)

class Embedding:
    def __init__(self, vocab_size, embed_size):
        self.weight = np.random.randn(vocab_size, embed_size) * 0.02
        self.x = None

    def forward(self, x):
        self.x = x
        return self.weight[x]

    def backward(self, dout):
        self.dW = np.zeros_like(self.weight)
        np.add.at(self.dW, self.x, dout)
        return None

class LayerNorm:
    def __init__(self, ndim, eps=1e-5):
        self.eps = eps
        self.gamma = np.ones(ndim)
        self.beta = np.zeros(ndim)

    def forward(self, x):
        mu = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mu) / np.sqrt(var + self.eps)
        self.cache = (x_norm, mu, var)
        return self.gamma * x_norm + self.beta

    def backward(self, dout):
        return dout * self.gamma

In [5]:
class FeedForward:
    def __init__(self, n_embd):
        self.c_fc = Linear(n_embd, 4 * n_embd)
        self.c_proj = Linear(4 * n_embd, n_embd)

    def forward(self, x):
        self.x_mid = self.c_fc.forward(x)
        return self.c_proj.forward(gelu(self.x_mid))

    def backward(self, dout):
        dout = self.c_proj.backward(dout)
        dout = gelu_backward(self.x_mid, dout)
        return self.c_fc.backward(dout)

class CausalSelfAttention:
    def __init__(self, n_embd, n_head):
        self.n_head = n_head
        self.head_size = n_embd // n_head
        self.c_attn = Linear(n_embd, 3 * n_embd)
        self.c_proj = Linear(n_embd, n_embd)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn.forward(x)
        q, k, v = np.split(qkv, 3, axis=-1)

        q = q.reshape(B, T, self.n_head, self.head_size).transpose(0, 2, 1, 3)
        k = k.reshape(B, T, self.n_head, self.head_size).transpose(0, 2, 1, 3)
        v = v.reshape(B, T, self.n_head, self.head_size).transpose(0, 2, 1, 3)

        mask = np.tril(np.ones((T, T))) == 0
        att = np.matmul(q, k.transpose(0, 1, 3, 2)) / np.sqrt(self.head_size)
        att[:, :, mask] = float('-inf')
        att = softmax(att, axis=-1)

        y = np.matmul(att, v).transpose(0, 2, 1, 3).reshape(B, T, C)
        return self.c_proj.forward(y)

    def backward(self, dout):
        dout = self.c_proj.backward(dout)
        return self.c_attn.backward(np.concatenate([dout, dout, dout], axis=-1))

class TransformerBlock:
    def __init__(self, n_embd, n_head):
        self.ln_1 = LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln_2 = LayerNorm(n_embd)
        self.mlp = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.attn.forward(self.ln_1.forward(x))
        x = x + self.mlp.forward(self.ln_2.forward(x))
        return x

    def backward(self, dout):
        dout = dout + self.mlp.backward(self.ln_2.backward(dout))
        dout = dout + self.attn.backward(self.ln_1.backward(dout))
        return dout

In [9]:
class GPT:
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer):
        self.block_size = block_size
        self.token_emb = Embedding(vocab_size, n_embd)
        self.pos_emb = Embedding(block_size, n_embd)
        self.blocks = [TransformerBlock(n_embd, n_head) for _ in range(n_layer)]
        self.ln_f = LayerNorm(n_embd)
        self.lm_head = Linear(n_embd, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = np.arange(0, T, dtype=np.int32)
        x = self.token_emb.forward(idx) + self.pos_emb.forward(pos)

        for block in self.blocks:
            x = block.forward(x)

        return self.lm_head.forward(self.ln_f.forward(x))

    def backward(self, dlogits):
        dx = self.ln_f.backward(self.lm_head.backward(dlogits))

        for block in reversed(self.blocks):
            dx = block.backward(dx)

        self.token_emb.backward(dx)

        # FIX: Sum the gradient across the Batch dimension for pos_emb!
        self.pos_emb.backward(np.sum(dx, axis=0))

def generate(model, tokenizer, idx, max_new_tokens, temperature=1.0):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.block_size:]
        logits = model.forward(idx_cond)[:, -1, :] / temperature
        probs = softmax(logits, axis=-1)
        idx_next = np.array([[np.random.choice(len(probs[0]), p=probs[0])]])
        idx = np.concatenate((idx, idx_next), axis=1)
    return tokenizer.decode(idx[0].tolist())

In [7]:
def cross_entropy_loss(logits, targets):
    B, T, C = logits.shape
    logits_2d = logits.reshape(B * T, C)
    targets_1d = targets.reshape(B * T)

    shifted_logits = logits_2d - np.max(logits_2d, axis=1, keepdims=True)
    Z = np.sum(np.exp(shifted_logits), axis=1, keepdims=True)
    log_probs = shifted_logits - np.log(Z)

    loss = np.mean(-log_probs[np.arange(B * T), targets_1d])

    dlogits = np.exp(log_probs)
    dlogits[np.arange(B * T), targets_1d] -= 1
    dlogits = (dlogits / (B * T)).reshape(B, T, C)

    return loss, dlogits

class Adam:
    def __init__(self, model, lr=1e-3):
        self.lr = lr
        self.b1, self.b2, self.eps = 0.9, 0.999, 1e-8
        self.t, self.m, self.v = 0, {}, {}
        self.params = [model.lm_head]

    def step(self):
        self.t += 1
        for layer in self.params:
            if hasattr(layer, 'W'):
                if layer not in self.m:
                    self.m[layer] = np.zeros_like(layer.W)
                    self.v[layer] = np.zeros_like(layer.W)

                self.m[layer] = self.b1 * self.m[layer] + (1 - self.b1) * layer.dW
                self.v[layer] = self.b2 * self.v[layer] + (1 - self.b2) * (layer.dW ** 2)

                m_hat = self.m[layer] / (1 - self.b1 ** self.t)
                v_hat = self.v[layer] / (1 - self.b2 ** self.t)

                layer.W -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [12]:
# Setup
block_size = 8
tokenizer = CharTokenizer(text_data)
dataset = Dataset(text_data, tokenizer, block_size)

model = GPT(
    vocab_size=tokenizer.vocab_size,
    block_size=block_size,
    n_embd=32,
    n_head=4,
    n_layer=2
)

optimizer = Adam(model, lr=1e-3)

# Train Loop
print("Training...")
for step in range(15):
    X, Y = dataset.get_batch(batch_size=4)
    logits = model.forward(X)
    loss, dlogits = cross_entropy_loss(logits, Y)
    model.backward(dlogits)
    optimizer.step()
    print(f"Step {step} | Loss: {loss:.4f}")



Training...
Step 0 | Loss: 4.3591
Step 1 | Loss: 4.1818
Step 2 | Loss: 4.0977
Step 3 | Loss: 4.5456
Step 4 | Loss: 4.2730
Step 5 | Loss: 4.4026
Step 6 | Loss: 4.1700
Step 7 | Loss: 4.4254
Step 8 | Loss: 4.4911
Step 9 | Loss: 4.0669
Step 10 | Loss: 4.1856
Step 11 | Loss: 4.1027
Step 12 | Loss: 4.2613
Step 13 | Loss: 4.0978
Step 14 | Loss: 4.1514


In [14]:
# Generate
print("\nGenerating text...")
# Make sure the context string exists in your dataset.txt!
context = text_data[:6] if len(text_data) >= 6 else text_data
encoded_context = np.array([tokenizer.encode(context)])
generated = generate(model, tokenizer, encoded_context, max_new_tokens=20)
print(f"Output:\n{generated}")


Generating text...
Output:
First !Iaq n h?UJqjqTLgp$h
